# 01 — Projektumfang und Anforderungen

## Worum geht es?
Dieses Projekt untersucht die **Luftqualität in acht europäischen Großstädten** mit den Methoden
des Data Engineering. Im Mittelpunkt stehen drei Schadstoffe:

- **PM2.5** — Feinstaub (Partikel < 2,5 µm)
- **PM10** — Grobstaub (Partikel < 10 µm)
- **NO2** — Stickstoffdioxid (vor allem aus Verkehr)

**Betrachtete Städte:** Wien, Berlin, Paris, Madrid, Rom, Amsterdam, Warschau, Prag.

## Leitfrage
> Wie unterscheiden sich PM2.5-, PM10- und NO2-Werte zwischen ausgewählten europäischen Städten,
> und welchen vorsichtig interpretierbaren Kontext liefern städtische Metadaten (z. B. Bevölkerungsdichte)?

## Was wir mit dem Projekt zeigen wollen
Nicht eine möglichst komplexe Analyse, sondern ein **sauberes, nachvollziehbares Data-Engineering-Setup**:
Daten aus drei verschiedenartigen Quellen holen, über Kafka und Spark verarbeiten, in einer
Schichtenarchitektur speichern und am Ende eine ehrliche, deskriptive Ergebnisgeschichte erzählen.

## Die drei Datenquellen
Die Projektanforderung verlangt drei **unterschiedliche** Quelltypen. Wir erfüllen sie so:

| Quelle | Typ | Was sie liefert |
| --- | --- | --- |
| **EEA Downloads API** | Datei/Datenbank (Parquet → PostgreSQL bzw. Parquet) | Historische, gemessene Schadstoffwerte (ein Jahr) |
| **Wikipedia** | Web-Scraping (HTML) | Bevölkerung, Fläche, Bevölkerungsdichte je Stadt |
| **Open-Meteo Air Quality API** | REST-API (JSON) | Aktuelle Live-Werte als Kafka-/Spark-Nachweis |

## Aufbau: Medallion-Architektur (Bronze → Silver → Gold)
Die Daten durchlaufen drei Schichten mit klar getrennten Aufgaben:

- **Bronze — Rohdaten:** quelltreu gespeichert, nichts wird verändert (PostgreSQL/Parquet, HTML, JSON).
- **Silver — bereinigt:** validiert, einheitliches Schema, über `city_id` verknüpfbar (Parquet).
- **Gold — analysebereit:** aggregierte Tabellen und Rangfolgen, die Notebook `09` direkt visualisiert.

So bleibt jede Transformation nachvollziehbar und wiederholbar.

## Ablauf der Notebooks
Die Notebooks werden in Reihenfolge ausgeführt. Jedes ist eigenständig dokumentiert.

| NB | Inhalt |
| --- | --- |
| `00` | Infrastruktur hochfahren (Kafka, Spark, PostgreSQL) |
| `01` | Dieses Notebook: Umfang, Leitfrage, Anforderungen |
| `02` | Quellen- und Cluster-Erreichbarkeit prüfen |
| `03` | Stadtreferenz (`city_id`-Katalog) erstellen |
| `04` | EEA-Messwerte abrufen → Bronze → Silver (Tageswerte) |
| `05` | Wikipedia scrapen → Stadtmetadaten (Silver) |
| `06` | Open-Meteo abrufen → Kafka-Producer |
| `07` | Spark liest aus Kafka → Parquet (Bronze/Silver) |
| `08` | Gold-Tabellen und Datenqualität |
| `09` | Analyse, Visualisierung, Ergebnisgeschichte |
| `10` | Daten löschen und Infrastruktur herunterfahren |

## Abdeckung der Projektanforderungen
Die folgende Zelle hält fest, welches Notebook welche Pflichtanforderung erfüllt, und prüft,
dass keine Anforderung vergessen wurde.

In [ ]:
requirements = {
    "file_or_database_source": "04",   # EEA: Parquet/PostgreSQL
    "web_scraping": "05",              # Wikipedia
    "rest_api": "06",                  # Open-Meteo
    "kafka_producer": "06",            # Events nach Kafka
    "spark_reads_kafka": "07",         # Spark Structured Streaming
    "persistence_parquet": "03-08",    # Silver/Gold-Parquet
    "storytelling": "09",             # Ergebnisgeschichte
}

# Prüfen, dass jede Pflichtanforderung abgedeckt ist.
expected = {"file_or_database_source", "web_scraping", "rest_api",
            "kafka_producer", "spark_reads_kafka", "persistence_parquet", "storytelling"}
missing = expected - set(requirements)
assert not missing, f"Nicht zugeordnete Anforderungen: {missing}"
print("Alle Projektanforderungen sind einem Notebook zugeordnet:")
requirements

## Nicht-Ziele
Bewusst **nicht** Teil des Projekts: produktives Dashboard, Machine-Learning-Modell,
Orchestrierung (Airflow/dbt), Cloud-Deployment. Es geht um das Data-Engineering-Setup, nicht um Größe.

## Nächster Schritt
Notebook `02` ausführen — Erreichbarkeit der Quellen und der Infrastruktur prüfen.